In [3]:
import sys
print(sys.executable)
!{sys.executable} -m pip --version
!{sys.executable} -m pip show streamlit

c:\Trainings\capstone-data-science\ai-for-energy\.venv\Scripts\python.exe
pip 26.0 from c:\Trainings\capstone-data-science\ai-for-energy\.venv\Lib\site-packages\pip (python 3.11)



In [4]:
import sys
!{sys.executable} -m pip install streamlit==1.36.0 plotly==5.24.1


  Using cached streamlit-1.36.0-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached plotly-5.24.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached pillow-10.4.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl.metadata (592 bytes)
  Using cached pyarrow-23.0.1-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached rich-13.9.4-py3-none-any.whl.metadata (18 kB)
  Using cached tenacity-8.5.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached watchdog-4.0.2-py3-none-win_amd64.whl.metad


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

st.set_page_config(page_title="Energy Price Model Dashboard", layout="wide")
st.title("Energy Price Forecast Dashboard")

@st.cache_data
def load_data(path):
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp", "model", "y_true", "y_pred"]).copy()
    df["abs_error"] = (df["y_true"] - df["y_pred"]).abs()
    df["sq_error"] = (df["y_true"] - df["y_pred"]) ** 2
    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    return df

df = load_data("predictions.csv")

models = sorted(df["model"].unique())
sel_models = st.sidebar.multiselect("Modelle", models, default=models)

min_ts, max_ts = df["timestamp"].min(), df["timestamp"].max()
date_range = st.sidebar.date_input("Zeitraum", [min_ts.date(), max_ts.date()])

mask = df["model"].isin(sel_models)
mask &= df["timestamp"].dt.date >= date_range[0]
mask &= df["timestamp"].dt.date <= date_range[1]
f = df.loc[mask].copy()

if f.empty:
    st.warning("Keine Daten für die aktuelle Filterauswahl.")
    st.stop()

metrics = (
    f.groupby("model")
    .agg(
        n=("y_true", "size"),
        mae=("abs_error", "mean"),
        rmse=("sq_error", lambda x: np.sqrt(x.mean())),
        mape=("abs_error", lambda x: np.mean(x / np.maximum(np.abs(f.loc[x.index, "y_true"]), 1e-6)) * 100),
    )
    .sort_values("mae")
    .reset_index()
)

st.subheader("Model Ranking")
st.dataframe(metrics, use_container_width=True)

best_model = metrics.iloc[0]["model"]
fm = f[f["model"] == best_model].sort_values("timestamp")

c1, c2 = st.columns(2)
with c1:
    st.metric("Bestes Modell", best_model)
with c2:
    st.metric("MAE (best)", f"{metrics.iloc[0]['mae']:.3f}")

st.subheader(f"Forecast vs Actual ({best_model})")
fig_ts = px.line(
    fm, x="timestamp", y=["y_true", "y_pred"],
    labels={"value": "Price", "timestamp": "Time", "variable": "Series"}
)
st.plotly_chart(fig_ts, use_container_width=True)

st.subheader("Fehler nach Stunde")
err_hour = (
    f.groupby(["model", "hour"])["abs_error"]
    .mean()
    .reset_index()
)
fig_hour = px.line(err_hour, x="hour", y="abs_error", color="model", markers=True)
st.plotly_chart(fig_hour, use_container_width=True)

st.subheader("Residual Distribution")
f["residual"] = f["y_true"] - f["y_pred"]
fig_hist = px.histogram(f, x="residual", color="model", barmode="overlay", nbins=60)
st.plotly_chart(fig_hist, use_container_width=True)


2026-02-16 13:15:22.437 
  command:

    streamlit run c:\Trainings\capstone-data-science\ai-for-energy\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-16 13:15:22.437 No runtime found, using MemoryCacheStorageManager
2026-02-16 13:15:22.445 No runtime found, using MemoryCacheStorageManager


FileNotFoundError: [Errno 2] No such file or directory: 'predictions.csv'